# Al: complete electronic-to-ionic workflow

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/otter-hed/otter/blob/main/notebooks/00-otter_intro.ipynb)

Calculate aluminium with the same settings, plots and colours as the [HTML example](https://otter-hed.github.io/otter/gen_examples/plot_al_full_workflow.html). All calculation and plotting code is included below.

## Install Otter

Otter 0.3.1 provides the orbital interfaces and direct `zstar` access used below. The version is pinned to reproduce this example.

In [ ]:
%pip install -q "otter-hed==0.3.1"

In [ ]:
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np

import otter
from otter import PlasmaWorkflowConfig, solve_plasma_workflow
from otter.plotting import grid_figsize, style_context

print(f"Otter {otter.__version__}")

## Input

Aluminium at $\rho=8.1\,\mathrm{g\,cm^{-3}}$ and $T_e=T_i=1\,\mathrm{eV}$. Only the physical inputs are set; all numerical controls use Otter's defaults.

In [ ]:
ELEMENT = "Al"
RHO_G_CC = 8.1
TE_EV = 1.0
TI_EV = 1.0
MULTIPLY_WAVEFUNCTION_BY_FD = False

## Calculate

Otter prints the full/external SCF progress (`d_n`, `d_v`). The wall time below excludes installation and plotting. Here $\bar Z=Z-\int 4\pi r^2 n_{\rm ion}(r)\,dr$ uses the complete radial grid, and $Z^*=n_0/n_i$.

In [ ]:
config = PlasmaWorkflowConfig(
    elements=[ELEMENT],
    rho_g_cc=RHO_G_CC,
    temperature_ev=TE_EV,
    ion_temperature_ev=TI_EV,
)
calculation_started = perf_counter()
result = solve_plasma_workflow(config)
calculation_elapsed = perf_counter() - calculation_started
electronic = result["electronic"]["result"]
ion = result["ion"]
Zbar = electronic["zbar_partition"]
Zstar = electronic["zstar"]

print(f"mu={electronic['mu']:.8f} Ha")
print(f"Zbar={Zbar:.8f}, Zstar={Zstar:.8f}")
print(f"HNC residual={ion['hnc_best_residual']:.3e}")
print(f"Calculation wall time: {calculation_elapsed:.2f} s ({calculation_elapsed / 60:.2f} min)")

## Bound orbitals

The wavefunctions are unweighted by default. Ionic density components include FD occupation, $M(E)$ and $f_{\rm cut}(r)$; their sum gives $n_{\rm ion}(r)$. Their Fourier transforms sum to $f(k)$. The real-space panels display $4\pi r^2$ times the corresponding profiles.

In [ ]:
from otter import bound_wavefunctions, ion_orbital_form_factors

energies = np.asarray(electronic["bound_energy_ha"])
valid = np.isfinite(energies) & (energies < electronic["bound_energy_cut_ha"])
li, ni = np.nonzero(valid)
angular = np.asarray(electronic["bound_l_list"], dtype=int)[li]
principal = ni + angular + 1
r_wave, r, k = electronic["r_bound"], electronic["r"], ion["k"]
multiply_fd = MULTIPLY_WAVEFUNCTION_BY_FD
wave = bound_wavefunctions(electronic, multiply_fd=multiply_fd)[valid]
density = electronic["ion_orbital_density_r"][valid]
factors = ion_orbital_form_factors(electronic, r=ion["r"], k=ion["k"])[valid]
total_density, r_ws = electronic["n_ion"], float(electronic["r_ws"])
r_max, k_max = 5.0, 10.0
electronic_title = (
    rf"{config.elements[0]}, $\rho={config.rho_g_cc:g}$ g cm$^{{-3}}$, "
    rf"$T_e={config.temperature_ev:g}$ eV, $\mu={electronic['mu']:.5f}$ Ha"
)
wave_mask, r_mask, k_mask = r_wave <= r_max, r <= r_max, k <= k_max
wave_shell, density_shell = 4 * np.pi * r_wave**2, 4 * np.pi * r**2
shell_letters = "spdfghiklm"
with style_context("thesis", palette="bing"):
    fig_orbitals, axes = plt.subplots(1, 3, figsize=grid_figsize(1, 3), layout="constrained")
    for i, (l, n) in enumerate(zip(angular, principal)):
        l, n = int(l), int(n)
        label = f"{n}{shell_letters[l]}" if l < len(shell_letters) else f"n={n}, l={l}"
        axes[0].plot(r_wave[wave_mask], (wave_shell * wave[i])[wave_mask], label=label)
        axes[1].plot(r[r_mask], (density_shell * density[i])[r_mask], label=label)
        axes[2].plot(k[k_mask], factors[i, k_mask], label=label)
    axes[1].plot(r[r_mask], (density_shell * total_density)[r_mask], color="black", ls="--", label="Total")
    axes[2].plot(k[k_mask], factors.sum(axis=0)[k_mask], color="black", ls="--", label="Total")
    axes[0].set(xlabel=r"$r$ [$a_B$]", xlim=(-0.25, r_max),
                ylabel=(r"$4\pi r^2 f_{\rm FD}R_{nl}(r)$" if multiply_fd else r"$4\pi r^2 R_{nl}(r)$") + r" [$a_B^{1/2}$]",
                title="FD-weighted wavefunctions" if multiply_fd else "Unweighted wavefunctions")
    axes[1].set(xlabel=r"$r$ [$a_B$]", xlim=(-0.25, r_max),
                ylabel=r"$4\pi r^2 n^{\rm ion}_{nl}(r)$ [$a_B^{-1}$]", title="Ion-orbital densities")
    axes[2].set(xlabel=r"$k$ [$a_B^{-1}$]", xlim=(0, k_max),
                ylabel=r"$f_{nl}(k)=n^{\rm ion}_{nl}(k)$", title="Ion-orbital form factors")
    for ax in axes[:2]:
        ax.axvline(r_ws, color="0.6", ls="--", lw=1.0, label=r"$R_{\rm WS}$")
    for ax in axes:
        if ax.lines:
            ax.legend()
    fig_orbitals.suptitle(electronic_title)
plt.show()

## Electronic structure

Electronic densities, full/external effective potentials, and full-AA potential components. The ionic density is $n_{\rm ion}$; the pseudoatom and screening densities satisfy $n_{\rm PA}=n_{\rm full}-n_{\rm ext}$ and $n_{\rm scr}=n_{\rm PA}-n_{\rm ion}$.

In [ ]:
r_e = np.asarray(electronic["r"])
e_mask = r_e <= 8.0
shell = 4.0 * np.pi * r_e**2
r_ws = float(electronic["r_ws"])
electronic_title = (
    rf"{config.elements[0]}, $\rho={config.rho_g_cc:g}$ g cm$^{{-3}}$, "
    rf"$T_e={config.temperature_ev:g}$ eV, $\mu={electronic['mu']:.5f}$ Ha"
)

with style_context("thesis", palette="bing"):
    fig_electronic, (ax_density, ax_potential, ax_components) = plt.subplots(
        1, 3, figsize=grid_figsize(1, 3), layout="constrained"
    )
    for key, label in (
        ("n_full", r"$n^{\rm full}$"), ("n_ion", r"$n^{\rm ion}$"),
        ("n_ext", r"$n^{\rm ext}$"), ("n_pa", r"$n^{\rm PA}$"),
        ("n_scr", r"$n^{\rm scr}$"), ("n0", r"$n_0$"),
    ):
        ax_density.plot(r_e[e_mask], (shell * electronic[key])[e_mask], label=label)
    ax_density.axvline(r_ws, color="0.25", ls=":", lw=1.1, label=r"$R_{\rm WS}$")
    ax_density.set(
        xlabel=r"$r$ [$a_B$]", ylabel=r"$4\pi r^2n(r)$ [$a_B^{-1}$]",
        xlim=(-0.5, 8.0), ylim=(-1.0, 15.0), title="Electronic densities",
    )
    ax_density.legend(ncol=2)

    for key, label in (
        ("v_full", r"$V_{\rm eff}^{\rm full}$"), ("v_ext", r"$V_{\rm eff}^{\rm ext}$"),
    ):
        ax_potential.plot(r_e[e_mask], electronic[key][e_mask], lw=2.0, label=label)
    ax_potential.set(
        xlabel=r"$r$ [$a_B$]", ylabel=r"$V(r)$ [Ha]",
        xlim=(-0.5, 5.0), ylim=(-1.0, 1.0), title="Effective potentials",
    )

    for key, label, linestyle, width in (
        ("v_full", r"$V_{\rm eff}^{\rm full}$", "-", 2.0),
        ("v_H", r"$V_{\rm H}$", "--", 1.5),
        ("v_xc", r"$V_{\rm xc}$", "--", 1.5),
        ("v_nuc", r"$V_{\rm nuc}$", "--", 1.5),
    ):
        ax_components.plot(
            r_e[e_mask], electronic[key][e_mask], ls=linestyle, lw=width, label=label
        )
    ax_components.set(
        xlabel=r"$r$ [$a_B$]", ylabel=r"$V(r)$ [Ha]",
        xlim=(-0.5, 8.0), ylim=(-8.0, 8.0), title="Full-AA potential components",
    )
    for ax in (ax_potential, ax_components):
        ax.axhline(0.0, color="0.5", ls=":", lw=0.9)
        ax.axvline(r_ws, color="0.25", ls=":", lw=1.1, label=r"$R_{\rm WS}$")
    ax_potential.legend()
    ax_components.legend(ncol=2)

    fig_electronic.suptitle(electronic_title)
plt.show()

## Pseudoatom to ion structure

Screening cloud, pair interactions, and ionic correlations use the completed workflow. The Rayleigh weight per ion is $W_R(k)=|f(k)+q(k)|^2S_{ii}(k)$.

In [ ]:
k = np.asarray(ion["k"])
r = np.asarray(ion["r"])
k_mask = k <= 8.0
r_mask = r <= 12.0

with style_context("thesis", palette="bing"):
    fig_pipeline, (ax_q, ax_vk, ax_vr) = plt.subplots(
        1, 3, figsize=grid_figsize(1, 3), layout="constrained"
    )
    fig_ionic, (ax_g, ax_s, ax_w) = plt.subplots(
        1, 3, figsize=grid_figsize(1, 3), layout="constrained"
    )

    ax_q.plot(k[k_mask], ion["q_k"][k_mask])
    ax_q.set(title=r"$q(k)=n_{\rm scr}(k)$", xlabel=r"$k$ [$a_B^{-1}$]", ylabel="electrons")

    ax_vk.plot(k[k_mask], ion["vii_k"][k_mask])
    ax_vk.set(title=r"$V_{ii}(k)$", xlabel=r"$k$ [$a_B^{-1}$]", ylabel=r"Ha $a_B^3$")

    ax_vr.plot(r[r_mask], ion["vii_r"][r_mask])
    ax_vr.set(title=r"$V_{ii}(r)$", xlabel=r"$r$ [$a_B$]", ylabel="Ha", xlim=(-0.5, 12.0))

    ax_g.plot(r[r_mask], ion["gii_r"][r_mask])
    ax_g.axhline(1.0, color="0.5", lw=0.8, ls=":")
    ax_g.set(title=r"$g_{ii}(r)$", xlabel=r"$r$ [$a_B$]", ylabel=r"$g_{ii}(r)$", xlim=(-0.5, 12.0))

    ax_s.plot(k[k_mask], ion["sii_k"][k_mask])
    ax_s.axhline(1.0, color="0.5", lw=0.8, ls=":")
    ax_s.set(title=r"$S_{ii}(k)$", xlabel=r"$k$ [$a_B^{-1}$]", ylabel=r"$S_{ii}(k)$")

    weight = np.abs(ion["f_k"] + ion["q_k"])**2 * ion["sii_k"]
    ax_w.plot(k[k_mask], weight[k_mask])
    ax_w.set(title="Rayleigh weight", xlabel=r"$k$ [$a_B^{-1}$]", ylabel=r"$W_R(k)$")

    ionic_title = (
        rf"{config.elements[0]} pseudoatom/QOZ/HNC, $\rho={config.rho_g_cc:g}$ g cm$^{{-3}}$, "
        rf"$T_e={config.temperature_ev:g}$ eV, $T_i={config.ion_temperature_ev:g}$ eV"
    )
    fig_pipeline.suptitle(ionic_title)
    fig_ionic.suptitle(ionic_title)
plt.show()